# 🌤️ Module 7 (Advanced): Weather Dimension & Trip-Weather Correlation

## Overview

In this notebook we add a brand-new data source — **daily weather data for Oslo 2026** — and integrate it with our star schema to analyse how weather influences bike-sharing behaviour.

**What you'll learn:**
- Introduce a new data source into an existing star schema
- Build `gold.dim_weather` at the same daily grain as the trip fact's `date_key`
- Join weather directly to trip facts using `date_key`
- Run correlation queries between weather and trip volume/duration
- Identify "good biking weather" conditions

---

**Prerequisites:**
- Completed Module 3 (Gold Layer)
- `silver_trips`, `gold.fact_trips`, and `gold.dim_date` tables exist

---

## 🎓 Concept: Weather as a Fact-Linked Dimension

`gold.dim_weather` has one row per calendar day and uses the same `date_key` format as the trip fact table.

The important relationship is between the weather dimension and the fact table:

```text
gold.dim_weather ── date_key ──► gold.fact_trips
                                      │
                                      ├──► gold.dim_date
                                      ├──► gold.dim_time
                                      ├──► gold.dim_start_station
                                      └──► gold.dim_end_station
```

`gold.dim_date` is still useful for calendar attributes such as month, weekday, and weekend flags. But weather does not need to connect through `gold.dim_date`; it can join directly to the fact table using `date_key`.


## Data Source: Open-Meteo Historical API

We use the **Open-Meteo** free historical weather API (no API key required):  
`https://archive-api.open-meteo.com/v1/archive`

Parameters requested:
- `daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,wind_speed_10m_max,weather_code`
- Location: Oslo (lat=59.9139, lon=10.7522)
- Date range: 2026-01-01 to 2026-07-31 (adjust to match your data)

If network access is restricted in your environment, the notebook falls back to generating **synthetic weather data** using Oslo's historical climate normals.


## Step 1: Fetch Weather Data (with synthetic fallback)

In [ ]:
import requests
import json
from datetime import date, timedelta

# ── Configuration ──────────────────────────────────────────────────────────
LAT, LON    = 59.9139, 10.7522  # Oslo city centre
START_DATE  = "2026-01-01"
END_DATE    = "2026-07-31"

# ── Try Open-Meteo API ──────────────────────────────────────────────────────
API_URL = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={LAT}&longitude={LON}"
    f"&start_date={START_DATE}&end_date={END_DATE}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "precipitation_sum,wind_speed_10m_max,weather_code"
    "&timezone=Europe%2FOslo"
)

weather_data = None
try:
    r = requests.get(API_URL, timeout=15)
    r.raise_for_status()
    weather_data = r.json()
    print(f"✅ Fetched {len(weather_data['daily']['time'])} days from Open-Meteo API")
except Exception as e:
    print(f"⚠️  API unavailable ({e}). Using synthetic data instead.")


In [ ]:
import random
from datetime import date, timedelta

# ── Synthetic weather generator ─────────────────────────────────────────────
# Based on Oslo climate normals (source: met.no)
MONTHLY_CLIMATE = {
    #  month: (mean_temp, temp_std, precip_mean, wind_mean)
     1: (-3.0,  4.0, 50,  5.5),
     2: (-3.0,  4.5, 35,  5.2),
     3: ( 1.5,  4.0, 45,  5.0),
     4: ( 6.5,  4.0, 40,  4.8),
     5: (12.0,  4.0, 50,  4.5),
     6: (16.5,  3.5, 60,  4.0),
     7: (19.0,  3.0, 70,  3.8),
     8: (18.5,  3.0, 85,  3.8),
     9: (13.0,  3.5, 80,  4.2),
    10: ( 7.5,  3.5, 90,  4.8),
    11: ( 2.0,  3.5, 70,  5.2),
    12: (-1.5,  4.0, 55,  5.5),
}

# WMO weather codes: 0=clear, 1-3=cloudy, 51-67=rain/drizzle, 71-77=snow, 80-82=showers
def synthetic_weather_code(temp, precip):
    if precip > 5:
        return 61 if temp > 2 else 71   # rain or snow
    if precip > 1:
        return 51 if temp > 2 else 73   # drizzle or light snow
    return random.choice([0, 1, 2, 3])  # clear to cloudy

if weather_data is None:
    rng = random.Random(42)  # fixed seed for reproducibility
    start = date.fromisoformat(START_DATE)
    end   = date.fromisoformat(END_DATE)
    days  = [(start + timedelta(d)) for d in range((end - start).days + 1)]

    rows = []
    for d in days:
        m_temp, m_std, m_precip, m_wind = MONTHLY_CLIMATE[d.month]
        t_mean  = round(rng.gauss(m_temp,  m_std),  1)
        t_max   = round(t_mean + abs(rng.gauss(3, 1.5)), 1)
        t_min   = round(t_mean - abs(rng.gauss(3, 1.5)), 1)
        precip  = round(max(0, rng.gauss(m_precip / 30, m_precip / 20)), 1)
        wind    = round(max(0, rng.gauss(m_wind * 10, 5)), 1)
        wcode   = synthetic_weather_code(t_mean, precip)
        rows.append((d.isoformat(), t_max, t_min, t_mean, precip, wind, wcode))

    weather_rows = rows
    print(f"✅ Generated {len(rows)} days of synthetic weather data")
else:
    d = weather_data["daily"]
    weather_rows = list(zip(
        d["time"],
        d["temperature_2m_max"],
        d["temperature_2m_min"],
        d["temperature_2m_mean"],
        d["precipitation_sum"],
        d["wind_speed_10m_max"],
        d["weather_code"],
    ))
    print(f"✅ Parsed {len(weather_rows)} days from API response")

print(f"Sample row: {weather_rows[0]}")

## Step 2: Load Weather Data into a Staging Table

In [ ]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

schema = StructType([
    StructField("full_date_str",        StringType(),  True),
    StructField("temp_max_celsius",     DoubleType(),  True),
    StructField("temp_min_celsius",     DoubleType(),  True),
    StructField("temp_mean_celsius",    DoubleType(),  True),
    StructField("precipitation_mm",     DoubleType(),  True),
    StructField("wind_speed_kmh",       DoubleType(),  True),
    StructField("weather_code",         IntegerType(), True),
])

rows = [Row(**{
    "full_date_str":     r[0],
    "temp_max_celsius":  float(r[1]) if r[1] is not None else None,
    "temp_min_celsius":  float(r[2]) if r[2] is not None else None,
    "temp_mean_celsius": float(r[3]) if r[3] is not None else None,
    "precipitation_mm":  float(r[4]) if r[4] is not None else 0.0,
    "wind_speed_kmh":    float(r[5]) if r[5] is not None else 0.0,
    "weather_code":      int(r[6])   if r[6] is not None else 0,
}) for r in weather_rows]

staging_df = spark.createDataFrame(rows, schema=schema)
staging_df.createOrReplaceTempView("stg_weather")
print(f"Staged {staging_df.count()} rows")

## Step 3: Create `gold.dim_weather` (Daily Weather Dimension)

In [ ]:
%%sql
-- ============================================================
-- DIMENSION TABLE: dim_weather
-- Grain : one row per calendar day
-- Joins directly to fact tables via date_key (YYYYMMDD INT)
-- ============================================================
CREATE OR REPLACE TABLE gold.dim_weather
USING DELTA
AS
SELECT
    -- Daily key used to join directly to fact tables
    CAST(DATE_FORMAT(TO_DATE(full_date_str, 'yyyy-MM-dd'), 'yyyyMMdd') AS INT) AS date_key,
    TO_DATE(full_date_str, 'yyyy-MM-dd')    AS full_date,

    -- Temperature
    temp_max_celsius,
    temp_min_celsius,
    temp_mean_celsius,
    (temp_max_celsius - temp_min_celsius)   AS temp_range_celsius,

    -- Precipitation
    precipitation_mm,

    -- Wind
    wind_speed_kmh,

    -- WMO weather code → human-readable condition
    -- Reference: https://open-meteo.com/en/docs#weathervariables
    CASE
        WHEN weather_code = 0              THEN 'Clear sky'
        WHEN weather_code IN (1, 2, 3)    THEN 'Mainly clear / Partly cloudy'
        WHEN weather_code IN (45, 48)     THEN 'Foggy'
        WHEN weather_code BETWEEN 51  AND 57  THEN 'Drizzle'
        WHEN weather_code BETWEEN 61  AND 67  THEN 'Rain'
        WHEN weather_code BETWEEN 71  AND 77  THEN 'Snow'
        WHEN weather_code BETWEEN 80  AND 82  THEN 'Rain showers'
        WHEN weather_code BETWEEN 85  AND 86  THEN 'Snow showers'
        WHEN weather_code IN (95, 96, 99) THEN 'Thunderstorm'
        ELSE 'Other'
    END                                     AS weather_condition,

    -- Simplified category for grouping
    CASE
        WHEN weather_code = 0              THEN 'Sunny'
        WHEN weather_code IN (1, 2, 3)    THEN 'Cloudy'
        WHEN weather_code BETWEEN 51  AND 82  THEN 'Rainy'
        WHEN weather_code BETWEEN 71  AND 77  THEN 'Snowy'
        ELSE 'Other'
    END                                     AS weather_category,

    -- 🚴 Good biking weather: warm enough, not too windy, little rain
    CASE
        WHEN temp_mean_celsius > 10
         AND precipitation_mm  < 3
         AND wind_speed_kmh    < 30
        THEN TRUE
        ELSE FALSE
    END                                     AS is_good_biking_weather,

    -- Temperature bucket for grouping
    CASE
        WHEN temp_mean_celsius < 0   THEN 'Freezing (<0°C)'
        WHEN temp_mean_celsius < 5   THEN 'Cold (0-5°C)'
        WHEN temp_mean_celsius < 10  THEN 'Cool (5-10°C)'
        WHEN temp_mean_celsius < 15  THEN 'Mild (10-15°C)'
        WHEN temp_mean_celsius < 20  THEN 'Warm (15-20°C)'
        ELSE                              'Hot (>20°C)'
    END                                     AS temp_bucket

FROM stg_weather
ORDER BY date_key

## Step 4: Verify `gold.dim_weather`

In [ ]:
%%sql
SELECT
    COUNT(*)                            AS total_days,
    MIN(full_date)                      AS first_date,
    MAX(full_date)                      AS last_date,
    SUM(CASE WHEN is_good_biking_weather THEN 1 ELSE 0 END) AS good_biking_days,
    ROUND(AVG(temp_mean_celsius), 1)    AS avg_temp,
    ROUND(AVG(precipitation_mm), 1)     AS avg_daily_precip_mm
FROM gold.dim_weather

In [ ]:
%%sql
-- Distribution by weather category
SELECT
    weather_category,
    COUNT(*)    AS days,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
FROM gold.dim_weather
GROUP BY weather_category
ORDER BY days DESC

## Step 5: Weather-Trip Correlation Queries

In [ ]:
%%sql
-- Daily trip count vs weather: weather joins directly to the fact on date_key
SELECT
    dw.full_date,
    dw.temp_mean_celsius,
    dw.precipitation_mm,
    dw.weather_category,
    dw.is_good_biking_weather,
    SUM(f.trip_count)                    AS trips,
    ROUND(AVG(f.duration_minutes), 1)    AS avg_duration_min
FROM gold.fact_trips f
JOIN gold.dim_weather dw ON f.date_key = dw.date_key
GROUP BY dw.full_date, dw.temp_mean_celsius, dw.precipitation_mm,
         dw.weather_category, dw.is_good_biking_weather
ORDER BY dw.full_date

In [ ]:
%%sql
-- Average trips per day by weather category
SELECT
    dw.weather_category,
    COUNT(DISTINCT f.date_key)           AS days_in_category,
    SUM(f.trip_count)                    AS total_trips,
    ROUND(SUM(f.trip_count) / COUNT(DISTINCT f.date_key), 0) AS avg_trips_per_day
FROM gold.fact_trips f
JOIN gold.dim_weather dw ON f.date_key = dw.date_key
GROUP BY dw.weather_category
ORDER BY avg_trips_per_day DESC

In [ ]:
%%sql
-- Good biking weather vs bad: trip volume comparison
SELECT
    dw.is_good_biking_weather,
    COUNT(DISTINCT f.date_key)           AS days,
    ROUND(SUM(f.trip_count) / COUNT(DISTINCT f.date_key), 0) AS avg_trips_per_day,
    MIN(daily_trips.trips)               AS min_trips_day,
    MAX(daily_trips.trips)               AS max_trips_day
FROM gold.fact_trips f
JOIN gold.dim_weather dw ON f.date_key = dw.date_key
JOIN (
    SELECT
        date_key,
        SUM(trip_count) AS trips
    FROM gold.fact_trips
    GROUP BY date_key
) daily_trips ON f.date_key = daily_trips.date_key
GROUP BY dw.is_good_biking_weather

In [ ]:
%%sql
-- Trip volume by temperature bucket
SELECT
    dw.temp_bucket,
    COUNT(DISTINCT f.date_key)           AS days,
    ROUND(SUM(f.trip_count) / COUNT(DISTINCT f.date_key), 0) AS avg_trips_per_day,
    ROUND(AVG(dw.temp_mean_celsius), 1)  AS avg_temp
FROM gold.fact_trips f
JOIN gold.dim_weather dw ON f.date_key = dw.date_key
GROUP BY dw.temp_bucket
ORDER BY avg_temp

In [ ]:
%%sql
-- Weekday vs weekend weather sensitivity
SELECT
    dd.is_weekend,
    dw.is_good_biking_weather,
    COUNT(DISTINCT f.date_key)           AS days,
    ROUND(SUM(f.trip_count) / COUNT(DISTINCT f.date_key), 0) AS avg_trips_per_day
FROM gold.fact_trips f
JOIN gold.dim_weather dw ON f.date_key = dw.date_key
JOIN gold.dim_date dd ON f.date_key = dd.date_key
GROUP BY dd.is_weekend, dw.is_good_biking_weather
ORDER BY dd.is_weekend, dw.is_good_biking_weather

## 📌 Key Takeaways

- `gold.dim_weather` is a daily weather dimension that joins directly to `gold.fact_trips` using `date_key`
- `gold.dim_date` still provides calendar context, but weather does not need to connect through it
- The **Open-Meteo API** is free, no-auth, and covers historical data worldwide — ideal for learning projects
- The synthetic fallback ensures the notebook always runs even without internet access
- Weather data can dramatically change analytical conclusions: understanding that rainy Mondays have fewer trips is only possible after connecting weather to trip facts